<a href="https://colab.research.google.com/github/Nirzaree/STAC-spec/blob/stac-spec-common/notebooks/generate_stac_pan_india_vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:

%pip install --quiet geopandas fiona

%pip install rasterio

%pip install pystac



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os

import json
import xml.etree.ElementTree as ET
import datetime
import requests
from io import BytesIO

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import shape, mapping, MultiPolygon, Polygon,box
from IPython.display import Image, display


import numpy as np
import pystac
from pystac.extensions.table import TableExtension
from pystac import CatalogType
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension


### **Used ijson python package for reading the large GEOJSON files and get the metadata **

In [ ]:
pip install ijson

In [ ]:
import datetime
from pystac import Item
import pandas as pd
import geopandas as gpd
import os
import ijson
import json
from shapely.geometry import shape, mapping

In [ ]:
# #geojson_filepath = '/content/drive/MyDrive/Pan_india/pan_india_drainage_lines.geojson'
# geojson_filepath = '/content/drive/MyDrive/Pan_india/Microwatershed_boundries_v2.geojson'

# geojson_filepath = "/content/drive/MyDrive/CopyofPAN_india_layers.csv"

geojson_filepath = "/content/drive/MyDrive/PAN_india_layers_vectors.csv"

STAC_SAVE_DIR = "/content/drive/MyDrive/STAC_spec_vector"
SUB_COLLECTIONS = {}

In [ ]:
df = pd.read_csv(geojson_filepath)

print("DataFrame loaded successfully")
print(df.head())

In [ ]:
try:
    # Define COLUMN_DESC_DF globally for use in run_stac_generation
    COLUMN_DESC_DF = pd.read_csv('/content/drive/MyDrive/column_descriptions.csv')
    print("Column descriptions loaded successfully.")
except FileNotFoundError:
    print("WARNING: 'column_descriptions.csv' not found. Table extension will be empty.")
    COLUMN_DESC_DF = pd.DataFrame({'layer_name': [], 'column_name': [], 'column_name_description': []})

In [ ]:
def create_root_and_collection():
    root_catalog = pystac.Catalog(
        id="PANindia",
        title="STAC Catalog for PAN India Vector Layers",
        description="Root catalog for PAN India vector datasets."
    )

    panindia_collection = pystac.Collection(
        id="panindia-vector-layers",
        title="CoREstack Pan India Vector Layers",
        description="A collection of various vector layers for Pan India.",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[68, 8, 98, 37]]),
            temporal=pystac.TemporalExtent([[datetime.datetime(2005, 1, 1), datetime.datetime(2025, 12, 31)]])
        ),
        license="https://spdx.org/licenses/CC-BY-4.0.html",
        providers=[
            pystac.Provider(
                name="CoREstack",
                roles=[
                    pystac.ProviderRole.PRODUCER,
                    pystac.ProviderRole.PROCESSOR,
                    pystac.ProviderRole.HOST
                ],
                url="https://core-stack.org/"
            )
        ],
        keywords=["vector", "spatial", "CoREstack"]
    )

    root_catalog.add_child(panindia_collection)
    return root_catalog, panindia_collection


In [ ]:
def create_sub_collection(title, parent_collection):

    title_str = str(title) if title is not None else "unknown"
    collection_id = title_str.lower().replace(' ', '-').replace(':', '').replace('/', '-')

    if collection_id not in SUB_COLLECTIONS:
        print(f"Creating new Sub-Collection: {title_str}")
        sub_collection = pystac.Collection(
            id=collection_id,
            title=title_str,
            description=f"Sub-collection for {title_str}",
            extent=parent_collection.extent,
            providers=parent_collection.providers
        )
        SUB_COLLECTIONS[collection_id] = sub_collection
    return SUB_COLLECTIONS[collection_id]


In [ ]:
def get_feature_geometry(geojson_filepath):

    with open(geojson_filepath, 'r') as f:
        for item in ijson.items(f, 'features.item', use_float=True):
            geom = item.get("geometry")
            props = item.get("properties", {})
            shapely_geom = shape(geom)
            minx, miny, maxx, maxy = shapely_geom.bounds
            return geom, [minx, miny, maxx, maxy]

In [ ]:

def get_first_feature_properties(geojson_filepath):

    with open(geojson_filepath, 'r') as f:
        for item in ijson.items(f, 'features.item.properties', use_float=True):
            properties_dict = {
                key: {
                    "value": value,
                    "type": type(value).__name__
                }
                for key, value in item.items()
            }
            break

    return {
        "properties": properties_dict
    }


In [ ]:
def generate_vector_stac(layer_name, file_path, column_desc_df):


    print(f"Processing vector layer: {layer_name}")

    item_id = layer_name.replace(" ", "_").replace("/", "_").replace("-", "_").lower()

    footprint, bbox = get_feature_geometry(file_path)


    properties_dict = get_first_feature_properties(file_path)["properties"]


    vector_gdf_dtypes = pd.DataFrame(
        [(key, value['type']) for key, value in properties_dict.items()],
        columns=['column_name', 'column_dtype']
    )

    try:
        vector_item = pystac.Item(
            id=item_id,
            geometry=footprint,
            bbox=bbox,
            datetime=datetime.datetime.now(datetime.timezone.utc),
            properties={"type": "vector", "description": f"Vector data for {layer_name}"}
        )

        proj_ext = ProjectionExtension.ext(vector_item, add_if_missing=True)
        proj_ext.epsg = 4326


        layer_filter_name = item_id


        vector_desc_filtered_df = column_desc_df[column_desc_df['layer_name'] == layer_filter_name]


        vector_merged_df = vector_gdf_dtypes.merge(
            vector_desc_filtered_df[['column_name', 'column_name_description']],
            on='column_name',
            how='left'
        ).fillna('')

        vector_merged_df.rename(columns={'column_name_description':'column_description'}, inplace=True)
        print(f"Schema merged for {layer_name}. Found {len(vector_merged_df[vector_merged_df['column_description'] != ''])} descriptions.")


        table_ext = TableExtension.ext(vector_item, add_if_missing=True)
        table_ext.columns = [
            {
                "name": row['column_name'],
                "type": str(row['column_dtype']),
                "description" : row['column_description']
            }
            for ind, row in vector_merged_df.iterrows()
        ]


        vector_item.add_asset("data", pystac.Asset(
            href=geojson_filepath,
            media_type=pystac.MediaType.GEOJSON,
            roles=["data"],
            title="Vector GeoJSON File"
        ))

        return vector_item

    except Exception as e:
        print(f"Error generating STAC Item for {layer_name}: {e}")
        return None

In [ ]:
def run_stac_generation(df, panindia_collection, root_catalog, column_desc_df):

    generated_items_count = 0
    current_collection = panindia_collection

    search_terms = ['Vector file path', 'GEE asset link', 'FilePath', 'URL', 'Link']
    file_path_col = None

    for term in search_terms:
        matches = [col for col in df.columns if term.lower() in col.lower()]
        if matches:

            file_path_col = matches[0]
            break

    if file_path_col is None:
        print("Fatal Error: Could not find a suitable column for file paths.")
        print("Available columns in your DataFrame are:")
        print(list(df.columns))
        print("Please rename your path column to include 'Vector file path' or 'GEE asset link'.")
        return

    print(f"File Path Column Identified: {file_path_col}")


    for idx, row in df.iterrows():
        layer_name = row["Layer Name"]
        file_path = row[file_path_col]

        if pd.isna(file_path) or str(file_path).strip() == '':
            current_collection = create_sub_collection(layer_name, panindia_collection)

            panindia_collection.add_child(current_collection)
            print(f"Switched context to Collection: {layer_name}")
            continue


        vector_item = generate_vector_stac(layer_name, file_path, column_desc_df)

        if vector_item:
            current_collection.add_item(vector_item)
            generated_items_count += 1
            print(f"Added Item {vector_item.id} to Collection {current_collection.id}")

    save_catalog(root_catalog, generated_items_count)

In [ ]:
def save_catalog(root_catalog, count):
    if count > 0:
        root_catalog.normalize_hrefs(STAC_SAVE_DIR)
        root_catalog.save(catalog_type=CatalogType.SELF_CONTAINED)
        print(f"STAC catalog saved")
        print(f"Total items generated: {count}")
    else:
        print("No items were generated.")


root_catalog, panindia_collection = create_root_and_collection()
run_stac_generation(df, panindia_collection, root_catalog, COLUMN_DESC_DF)